In [58]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath("../.."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df

from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold

In [74]:
cluster_df = pd.read_csv('../Indicator_Correlations/cluster.csv')  # includes 'indicator', 'cluster' columns
high_corr = pd.read_csv('../Indicator_Correlations/highly_correlated_pairs.csv')  # includes 'indicator_1', 'indicator_2'
low_corr = pd.read_csv('../Indicator_Correlations/low_corr_pairs.csv')  # includes 'indicator_1', 'indicator_2'
corr_matrix = pd.read_csv('../Indicator_Correlations/all_cors.csv')  # includes 'indicator_1', 'indicator_2'
ind_dynamics_df = pd.read_csv('../Indicator_Dynamics/indicator_dynamics.csv')  # includes 'r', 'indicator', volatility/zscore/autocorr columns
logistic_results_df = pd.read_csv('../Univariate_Predictive_Power/logistic_results_xgb.csv')  # includes AUC, logloss, etc.
high_level_cat = pd.read_csv('../Finalization/high_level_cat.csv') 

ticker = 'QQQ'
returns = [5, 10, 20, 30, 45, 60, 90]
lb = 20
df = final_df(ticker, returns, lb)
df = df.iloc[:-101].replace([np.inf, -np.inf], 0)

# Determining Velocity

In [79]:
# Initialize columns
high_level_cat = high_level_cat[high_level_cat['Type'] != 'Exclude']

for i, row in high_level_cat.iterrows():
    indicator = row['Indicator']

    if indicator not in df.columns:
        continue

    series = df[indicator]
    series = series / series.mean()

    # Skip non-numeric series
    if not np.issubdtype(series.dtype, np.number):
        continue

    velocity_25 = series.diff().abs().rolling(window=25).mean().mean()

    high_level_cat.at[i, 'velocity_25'] = velocity_25

# Define labels
labels = ['slow', 'moderate', 'fast']

# Global velocity binning
quantiles_global = high_level_cat['velocity_25'].quantile([0.33, 0.66])
bins_global = [-np.inf, quantiles_global[0.33], quantiles_global[0.66], np.inf]
high_level_cat['velocity_25_global'] = pd.cut(high_level_cat['velocity_25'], bins=bins_global, labels=labels)

def qbin(s):
    """Return a 3-level q-cut, or fall back to a single label."""
    s = s.dropna()
    if s.nunique() < 3:
        # not enough spread – label everything 'moderate'
        return pd.Series(['moderate'] * len(s), index=s.index)

    # Compute break-points; allow for ties
    q = np.quantile(s, [0, .33, .66, 1])
    # If duplicates remain after dropping, we fall back to qcut
    try:
        return pd.cut(s, bins=q, labels=labels, include_lowest=True,
                      duplicates='drop')
    except ValueError:                   # still too few bins
        return pd.qcut(s.rank(method='first'), 3, labels=labels)

# Type-local
high_level_cat['velocity_25_type_local'] = (
    high_level_cat.groupby('Type')['velocity_25'].transform(qbin)
)

# Category-local
high_level_cat['velocity_25_cat_local'] = (
    high_level_cat.groupby('Category')['velocity_25'].transform(qbin)
)

high_level_cat.head(25)

,Indicator,Type,Category,velocity_25,velocity_25_global,velocity_25_type_local,velocity_25_cat_local
0,10_EMA_100,Raw,trend_ratio,0.002547,slow,slow,fast
1,10_EMA_200,Raw,trend_ratio,0.002733,slow,slow,fast
2,10_EMA_25,Raw,trend_ratio,0.001532,slow,slow,moderate
3,10_EMA_50,Raw,trend_ratio,0.002184,slow,slow,moderate
4,10_ESMA_100,Raw,trend_ratio,0.002816,slow,slow,fast
5,10_ESMA_200,Raw,trend_ratio,0.002862,slow,slow,fast
6,10_ESMA_25,Raw,trend_ratio,0.002338,slow,slow,fast
7,10_ESMA_50,Raw,trend_ratio,0.002672,slow,slow,fast
8,10_SMA_100,Raw,trend_ratio,0.002808,slow,slow,fast
9,10_SMA_200,Raw,trend_ratio,0.002831,slow,slow,fast


# Add temporal consistency

In [80]:
# Set horizon days and number of bins
horizons = [5, 10, 20]
n_bins = 10

# Ensure Date and Year columns exist
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year

# Filter to raw indicators only
raw_indicators = high_level_cat.loc[high_level_cat['Type'] != 'Exclude', 'Indicator']

# Storage for results
results = []

for indicator in raw_indicators:
    if indicator not in df.columns:
        continue

    for r in horizons:
        return_col = f'Return_{r}'
        if return_col not in df.columns:
            continue

        # Drop NaNs for this indicator and return column
        temp = df[['Date', 'Year', indicator, return_col]].dropna().copy()

        try:
            # Global binning across all time
            temp['bin'] = pd.qcut(temp[indicator], q=n_bins, duplicates='drop')
        except:
            continue  # Skip if qcut fails

        # Yearly stats using shared bins
        yearly_bin_stats = {}
        for year, group in temp.groupby('Year'):
            bin_means = group.groupby('bin', observed=False)[return_col].mean()
            bin_weights = group['bin'].value_counts(normalize=True)
            baseline_return = group[return_col].mean()

            rel_lift = bin_means - baseline_return  # relative to year baseline

            yearly_bin_stats[year] = {
                'lift': bin_means,
                'dist': bin_weights,
                'rel_lift': rel_lift
            }

        # Year-over-year comparisons
        lift_consistencies = []
        dist_consistencies = []
        rel_lift_consistencies = []

        years = sorted(yearly_bin_stats.keys())
        for i in range(1, len(years)):
            y1, y2 = years[i - 1], years[i]

            lift1, lift2 = yearly_bin_stats[y1]['lift'], yearly_bin_stats[y2]['lift']
            dist1, dist2 = yearly_bin_stats[y1]['dist'], yearly_bin_stats[y2]['dist']
            rel1, rel2 = yearly_bin_stats[y1]['rel_lift'], yearly_bin_stats[y2]['rel_lift']

            shared_bins = lift1.index.intersection(lift2.index)
            if len(shared_bins) == 0:
                continue

            lift_diff = (lift1[shared_bins] - lift2[shared_bins]).abs().mean()
            dist_diff = (dist1[shared_bins] - dist2[shared_bins]).abs().mean()
            rel_lift_diff = (rel1[shared_bins] - rel2[shared_bins]).abs().mean()

            lift_consistencies.append(lift_diff)
            dist_consistencies.append(dist_diff)
            rel_lift_consistencies.append(rel_lift_diff)

        # Aggregate results
        results.append({
            'Indicator': indicator,
            'Horizon': r,
            'Lift_Consistency': np.mean(lift_consistencies) if lift_consistencies else np.nan,
            'Rel_Lift_Consistency': np.mean(rel_lift_consistencies) if rel_lift_consistencies else np.nan,
            'Dist_Consistency': np.mean(dist_consistencies) if dist_consistencies else np.nan
        })

# Final result
consistency_df = pd.DataFrame(results)

In [81]:
# Pivot for Lift_Consistency and Rel_Lift_Consistency
pivoted_df = consistency_df.pivot(
    index='Indicator',
    columns='Horizon',
    values=['Lift_Consistency', 'Rel_Lift_Consistency']
)

# Flatten the multi-index columns
pivoted_df.columns = [f"{metric}_{h}" for metric, h in pivoted_df.columns]

# Reset index
pivoted_df = pivoted_df.reset_index()

# Merge back the one column of Dist_Consistency (since it's constant per indicator)
dist_df = consistency_df[['Indicator', 'Dist_Consistency']].drop_duplicates(subset='Indicator')

# Final merge
pivoted_df = pivoted_df.merge(dist_df, on='Indicator', how='left')

pivoted_df

,Indicator,Lift_Consistency_5,Lift_Consistency_10,Lift_Consistency_20,Rel_Lift_Consistency_5,Rel_Lift_Consistency_10,Rel_Lift_Consistency_20,Dist_Consistency
0,100_EMA_200,0.265646,0.359861,0.413778,0.226201,0.304052,0.315947,0.124935
1,100_ESMA_200,0.319188,0.369419,0.404831,0.284495,0.320250,0.315106,0.124151
2,100_SMA_200,0.244900,0.319827,0.403362,0.221922,0.262658,0.320336,0.113562
3,10_EMA_100,0.293802,0.315460,0.352949,0.262690,0.243808,0.236361,0.099837
4,10_EMA_200,0.292052,0.311891,0.356783,0.251517,0.245005,0.278156,0.104726
...,...,...,...,...,...,...,...,...
505,vol_5_EA50,0.163151,0.204334,0.219407,0.138584,0.155480,0.132867,0.029069
506,vol_5_MA10,0.145834,0.164502,0.214805,0.125540,0.120451,0.123900,0.035328
507,vol_5_MA25,0.164717,0.192079,0.227051,0.137839,0.146572,0.143693,0.032253
508,vol_5_MA5,0.140719,0.171406,0.206593,0.118209,0.117873,0.115407,0.028137


In [82]:
merged_df = high_level_cat.merge(pivoted_df, on='Indicator', how='left')
merged_df.to_csv('tags_cons.csv')